In [66]:
import os
import qsprpred
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch

from qsprpred.data import QSPRDataset, RandomSplit
from qsprpred.data.descriptors.fingerprints import MorganFP
import pandas as pd
from qsprpred.data.descriptors.sets import RDKitDescs
from MolEval import MolEmb 

In [126]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold
from qsprpred.data.descriptors.sets import RDKitDescs

def load_datasets(path):
    dataset = QSPRDataset.fromTableFile(
    filename=path,
    store_dir="dataset_outputs/A2AR/data",
    name="CK1Dataset",
    target_props=[{"name": "Y", "task": "SINGLECLASS", "th": [0.5]}],
    random_state=42,
    smiles_col = 'Drug',
    sep=','
    )
    return dataset

class Dataset_creator():
    def __init__(self, model_names=['RoBERTa_ZINC'], corr_thrsh= 0.95):
        self.variance = VarianceThreshold(threshold=0.0)
        self.model_names = model_names
        self.corr_thrsh = corr_thrsh
        self.selected_indices = None
        
    def fit_transform(self, dataset):
        dataset.prepareDataset(feature_calculators=[MorganFP(radius=2, nBits=4096)], recalculate_features=True, shuffle=False)
        rdkit_descs = RDKitDescs()
        dataset.addDescriptors([rdkit_descs])
        display(dataset.X.shape)
        
        dataset.descriptorSets
        dataset_embs = self.create_embs(dataset)
        dataset_embs.index = dataset.X.index
        dataset.X = pd.concat([dataset.X, dataset_embs], axis=1)
        display(dataset.X.shape)
        
        dataset_no_var_np = self.variance.fit_transform(dataset.X)
        mask = self.variance.get_support()
        dataset_without_no_var = pd.DataFrame(
            dataset_no_var_np,
            columns=dataset.X.columns[mask],
            index=dataset.X.index)
        display(dataset_without_no_var.shape)

        
        dataset_without_high_corr = self.high_correlation(dataset_without_no_var)
        display(dataset_without_high_corr.shape)
        
        dataset.X = dataset_without_high_corr
        return dataset
        
    def transform(self, dataset):
        dataset.prepareDataset(feature_calculators=[MorganFP(radius=2, nBits=4096)], recalculate_features=True, shuffle=False)
        rdkit_descs = RDKitDescs()
        dataset.addDescriptors([rdkit_descs])
        display(dataset.X.shape)

        dataset.descriptorSets
        dataset_embs = self.create_embs(dataset)
        dataset_embs.index = dataset.X.index
        dataset.X = pd.concat([dataset.X, dataset_embs], axis=1)
        display(dataset.X.shape)

        dataset_no_var_np = self.variance.transform(dataset.X)
        mask = self.variance.get_support()
        dataset_without_no_var = pd.DataFrame(
            dataset_no_var_np,
            columns=dataset.X.columns[mask],
            index=dataset.X.index)
        display(dataset_without_no_var.shape)

        dataset_without_high_corr = dataset_without_no_var[self.selected_indices]
        display(dataset_without_high_corr.shape)

        dataset.X = dataset_without_high_corr
        return dataset

    def high_correlation(self, df: pd.DataFrame):
        corr_matrix = df.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        
        to_drop = [column for column in upper.columns if any(upper[column] > self.corr_thrsh)]
        self.selected_indices = df.columns.difference(to_drop)
        
        return df[self.selected_indices]

        
    def create_embs(self, dataset):
        dataset.df["SMILES"] = dataset.df["Drug"]
        final_emb = pd.DataFrame()
        for model_name in self.model_names:
            extractor = MolEmb.EmbeddingExtractor(model_name=model_name, df=dataset.df)
            new_emb, dataset.df = extractor.get_embeddings()
            display(type(new_emb))
            if final_emb.empty:
                final_emb = new_emb
            else:
                final_emb = pd.concat([final_emb, new_emb], axis=1)
        final_emb.columns = final_emb.columns.astype(str)
        display(final_emb)
        return final_emb
    


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X1_all = load_datasets("CK1/data/ck1_train_1")

X2_all = load_datasets("CK1/data/ck1_val_1")

X3_all = load_datasets("CK1/data/ck1_test_1")

In [4]:
cls = Dataset_creator(['Mol2Vec', 'RoBERTa_ZINC'])

In [5]:
X1_all = cls.fit_transform(X1_all)
X2_all = cls.transform(X2_all)
X3_all = cls.transform(X3_all)

(488, 4306)

pandas.core.frame.DataFrame

Some weights of RobertaModel were not initialized from the model checkpoint at entropy/roberta_zinc_480m and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


pandas.core.frame.DataFrame

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,2.516871,0.534238,-2.452955,7.686741,0.431078,-0.694065,-12.601948,-0.971291,7.476032,3.846928,...,-0.143628,0.515095,-0.074862,-0.102673,0.103030,0.130476,0.499113,-0.214322,-0.077564,-0.431823
1,6.052453,-2.925424,-2.070877,8.495329,1.301863,1.117611,-16.032846,-1.642392,9.346370,2.927347,...,0.051274,0.277398,-0.593216,0.042848,-0.011813,0.098258,0.729520,-0.701225,0.259042,-0.592370
2,5.744293,-7.785055,-3.891767,11.262932,0.251325,0.310802,-16.780500,-1.574069,10.337992,5.297069,...,-0.086957,0.134045,0.056459,-0.335765,0.136977,0.272289,1.180151,-0.274104,0.327263,-0.534128
3,5.114468,-4.830433,-5.447062,10.757203,0.077670,0.495645,-12.574245,-2.077898,7.660981,3.149048,...,-0.250937,0.206534,-0.223312,0.343322,-0.241193,0.214450,1.021452,-0.288710,-0.215809,-0.786897
4,2.010692,-1.409427,-3.583793,3.778501,1.039966,-1.023768,-7.496891,-0.705993,5.007909,2.194638,...,0.054637,0.027199,-0.383653,0.068650,0.585950,-0.025223,0.926432,0.213663,-0.226463,-0.451652
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483,3.387227,-3.312556,-3.568770,9.836091,-2.975115,-4.776808,-15.644789,0.402767,11.404202,5.728592,...,0.259191,-0.156011,-0.657177,0.090683,-0.131137,-0.044086,0.658195,-0.148066,-0.333033,-0.870234
484,2.365839,-5.633009,-3.388613,10.866866,-1.929445,0.710701,-15.617691,-1.977192,10.801051,3.905593,...,0.279238,0.092157,-0.210265,-0.054687,0.159311,0.148841,1.103577,-0.306556,0.361056,-0.596046
485,2.876224,-5.038481,-3.354145,4.159993,0.195354,-1.229711,-6.549203,0.585766,1.457568,0.045052,...,-0.088278,0.292331,-0.240328,-0.308141,-0.375836,-0.005479,0.483670,-0.645472,0.012842,-0.240922
486,3.581094,-6.064793,-2.769171,6.592866,-0.030812,-0.284410,-9.875506,0.379415,4.067562,0.976918,...,0.253126,0.356655,-0.030671,-0.108964,0.162669,0.056533,0.501798,-0.532637,-0.328418,-0.128803


...

pandas.core.frame.DataFrame

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,4.884953,-7.157942,-5.138814,13.006714,-2.730479,0.151554,-20.335842,-2.198932,17.312227,6.219527,...,0.010942,0.226891,-0.549037,0.132376,0.037968,0.605839,0.911198,-0.137456,0.129961,-0.967576
1,3.740307,-6.143758,-2.042488,10.055068,-1.950604,2.415757,-14.389182,-1.108633,10.755788,3.516679,...,-0.014316,0.385896,-0.545804,0.038958,0.287735,0.626203,1.166353,-0.095385,-0.005796,-0.972924
2,2.931786,-4.380294,-4.727276,12.547298,-0.920773,0.859071,-14.014235,-1.219353,10.044724,4.105293,...,0.153847,0.328493,0.075381,0.357536,-0.166716,0.382348,1.111882,-0.605728,0.704239,-0.395475
3,2.763307,-3.071845,-4.337036,7.157831,1.675385,-0.164037,-14.796813,0.702609,8.210071,3.552740,...,0.043420,0.275199,-0.554195,-0.329015,-0.197298,0.412721,0.569449,-0.312527,-0.098183,-0.531841
4,5.317841,-3.904930,-5.222112,10.552648,-0.299198,0.792047,-13.066146,-1.410062,9.210221,4.334833,...,-0.297946,0.241099,-0.360677,0.370727,-0.181313,0.198411,1.087878,-0.556719,-0.322968,-0.667929
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159,4.732400,-5.931112,-2.695037,7.379294,0.355767,1.439902,-9.418824,-0.047803,4.838798,0.808847,...,0.349798,0.256530,-0.020947,-0.335571,-0.034310,0.108234,0.765672,-0.501721,-0.253853,-0.082646
160,2.290391,-1.555030,-2.027585,5.427836,2.616212,1.212105,-11.124537,-0.277540,5.989487,-0.544919,...,0.083183,0.174510,-0.304669,-0.249486,0.221935,-0.219294,0.534043,-0.708747,-0.097672,0.005163
161,2.183160,-2.231052,-2.962017,7.712423,-0.278175,0.161704,-8.929571,-0.744787,6.846818,2.309243,...,-0.268816,0.230295,-0.299570,0.210466,0.212637,-0.155051,0.824359,-0.204696,0.053603,-0.599155
162,1.697251,-2.229853,-2.875325,7.633800,-0.657874,-0.268612,-8.608117,-0.441910,7.232983,2.815516,...,-0.106652,0.258743,-0.016806,-0.088005,0.264663,-0.236336,0.674913,-0.257922,0.189105,-0.678619


(164, 5374)

(164, 3757)

(164, 2400)

In [6]:
X1_all.X.to_csv("CK1/mod_data/X1.1")
X2_all.X.to_csv("CK1/mod_data/X2.1")
X3_all.X.to_csv("CK1/mod_data/X3.1")
X1_all.y.to_csv("CK1/mod_data/y1.1")
X2_all.y.to_csv("CK1/mod_data/y2.1")
X3_all.y.to_csv("CK1/mod_data/y3.1")

In [127]:
for i in range(2, 11):
    X1_all = load_datasets(f"CK1/data/ck1_train_{i}")

    X2_all = load_datasets(f"CK1/data/ck1_val_{i}")

    X3_all = load_datasets(f"CK1/data/ck1_test_{i}")
    cls = Dataset_creator(['Mol2Vec', 'RoBERTa_ZINC'])
    X1_all = cls.fit_transform(X1_all)
    X2_all = cls.transform(X2_all)
    X3_all = cls.transform(X3_all)

    X1_all.X.to_csv(f"CK1/mod_data/X1.{i}")
    X2_all.X.to_csv(f"CK1/mod_data/X2.{i}")
    X3_all.X.to_csv(f"CK1/mod_data/X3.{i}")
    X1_all.y.to_csv(f"CK1/mod_data/y1.{i}")
    X2_all.y.to_csv(f"CK1/mod_data/y2.{i}")
    X3_all.y.to_csv(f"CK1/mod_data/y3.{i}")

(488, 4306)

pandas.core.frame.DataFrame

Some weights of RobertaModel were not initialized from the model checkpoint at entropy/roberta_zinc_480m and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


pandas.core.frame.DataFrame

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,2.516871,0.534238,-2.452955,7.686741,0.431078,-0.694065,-12.601948,-0.971291,7.476032,3.846928,...,-0.143628,0.515095,-0.074862,-0.102673,0.103030,0.130476,0.499113,-0.214322,-0.077564,-0.431823
1,6.052453,-2.925424,-2.070877,8.495329,1.301863,1.117611,-16.032846,-1.642392,9.346370,2.927347,...,0.051274,0.277398,-0.593216,0.042848,-0.011813,0.098258,0.729520,-0.701225,0.259042,-0.592370
2,5.744293,-7.785055,-3.891767,11.262932,0.251325,0.310802,-16.780500,-1.574069,10.337992,5.297069,...,-0.086957,0.134045,0.056459,-0.335765,0.136977,0.272289,1.180151,-0.274104,0.327263,-0.534128
3,4.884953,-7.157942,-5.138814,13.006714,-2.730479,0.151554,-20.335842,-2.198932,17.312227,6.219527,...,0.010942,0.226891,-0.549037,0.132376,0.037968,0.605839,0.911198,-0.137456,0.129961,-0.967576
4,4.284906,-4.222572,-3.638047,6.127689,2.785984,-0.247923,-15.491641,0.266604,11.284584,2.270757,...,-0.160620,0.230485,-0.263681,-0.565322,0.152238,0.453253,1.078272,-0.581477,-0.351562,-0.548172
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483,3.387227,-3.312556,-3.568770,9.836091,-2.975115,-4.776808,-15.644789,0.402767,11.404202,5.728592,...,0.259191,-0.156011,-0.657177,0.090683,-0.131137,-0.044086,0.658195,-0.148066,-0.333033,-0.870234
484,2.365839,-5.633009,-3.388613,10.866866,-1.929445,0.710701,-15.617691,-1.977192,10.801051,3.905593,...,0.279238,0.092157,-0.210265,-0.054687,0.159311,0.148841,1.103577,-0.306556,0.361056,-0.596046
485,2.876224,-5.038481,-3.354145,4.159993,0.195354,-1.229711,-6.549203,0.585766,1.457568,0.045052,...,-0.088278,0.292331,-0.240328,-0.308141,-0.375836,-0.005479,0.483670,-0.645472,0.012842,-0.240922
486,3.581094,-6.064793,-2.769171,6.592866,-0.030812,-0.284410,-9.875506,0.379415,4.067562,0.976918,...,0.253126,0.356655,-0.030671,-0.108964,0.162669,0.056533,0.501798,-0.532637,-0.328418,-0.128803


...

pandas.core.frame.DataFrame

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,2.516871,0.534238,-2.452955,7.686741,0.431078,-0.694065,-12.601948,-0.971291,7.476032,3.846928,...,-0.143628,0.515095,-0.074862,-0.102673,0.103030,0.130476,0.499113,-0.214322,-0.077564,-0.431823
1,4.884953,-7.157942,-5.138814,13.006714,-2.730479,0.151554,-20.335842,-2.198932,17.312227,6.219527,...,0.010942,0.226891,-0.549037,0.132376,0.037968,0.605839,0.911198,-0.137456,0.129961,-0.967576
2,2.763307,-3.071845,-4.337036,7.157831,1.675385,-0.164037,-14.796813,0.702609,8.210071,3.552740,...,0.043420,0.275199,-0.554195,-0.329015,-0.197298,0.412721,0.569449,-0.312527,-0.098183,-0.531841
3,5.317841,-3.904930,-5.222112,10.552648,-0.299198,0.792047,-13.066146,-1.410062,9.210221,4.334833,...,-0.297946,0.241099,-0.360677,0.370727,-0.181313,0.198411,1.087878,-0.556719,-0.322968,-0.667929
4,2.975206,-6.491653,-3.839927,11.821246,-0.138459,1.003587,-17.784031,-2.597596,12.989882,3.524831,...,0.155326,0.251512,-0.274766,-0.096163,-0.007197,0.296403,0.830893,-0.167375,0.300061,-0.861038
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,4.732400,-5.931112,-2.695037,7.379294,0.355767,1.439902,-9.418824,-0.047803,4.838798,0.808847,...,0.349798,0.256530,-0.020947,-0.335571,-0.034310,0.108234,0.765672,-0.501721,-0.253853,-0.082646
158,2.183160,-2.231052,-2.962017,7.712423,-0.278175,0.161704,-8.929571,-0.744787,6.846818,2.309243,...,-0.268816,0.230295,-0.299570,0.210466,0.212637,-0.155051,0.824359,-0.204696,0.053603,-0.599155
159,4.080963,-6.359069,-5.269644,11.860126,-0.075333,3.266928,-18.994394,-1.355099,14.285904,4.428610,...,-0.180873,-0.124348,-0.549422,0.079085,0.075019,0.445874,0.769192,0.268825,0.107943,-0.835486
160,1.697251,-2.229853,-2.875325,7.633800,-0.657874,-0.268612,-8.608117,-0.441910,7.232983,2.815516,...,-0.106652,0.258743,-0.016806,-0.088005,0.264663,-0.236336,0.674913,-0.257922,0.189105,-0.678619


(162, 5374)

(162, 3797)

(162, 2436)

In [129]:
X1_all.X

,300,301,302,303,304,305,306,307,308,309,...,RDkit_fr_piperzine,RDkit_fr_pyridine,RDkit_fr_sulfide,RDkit_fr_sulfonamd,RDkit_fr_sulfone,RDkit_fr_thiazole,RDkit_fr_thiophene,RDkit_fr_unbrch_alkane,RDkit_fr_urea,RDkit_qed
QSPRID,,,,,,,,,,,,,,,,,,,,,
CK1Dataset_000,-0.183866,0.578967,0.161637,0.422820,-0.359473,-1.000239,0.778588,0.269597,0.671282,0.095040,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.389416
CK1Dataset_001,-0.526914,0.482046,0.214669,0.376727,-0.234737,-0.584075,0.579262,-0.091283,0.320457,-0.131275,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.351908
CK1Dataset_002,-0.527138,0.293057,0.296170,0.215663,-0.075322,-1.196916,0.347968,0.003652,0.324451,0.011451,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.312720
CK1Dataset_003,-0.255575,0.203898,0.418056,0.378673,-0.129638,-0.148893,0.742217,-0.251707,-0.097038,0.531059,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.441687
CK1Dataset_004,-0.687007,0.620376,0.137707,-0.190699,0.417505,-0.457825,0.332847,-0.721059,-0.052884,-0.030075,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.804595
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CK1Dataset_484,-0.223748,0.351416,-0.098620,0.191478,0.223344,-0.828098,0.288138,-0.034601,0.138777,0.187323,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.451445
CK1Dataset_485,0.170707,0.057631,0.364865,0.143474,0.144775,0.001812,0.526527,-0.131677,0.087674,0.613961,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.743722
CK1Dataset_486,-0.444729,0.174482,0.096636,0.235764,-0.291311,-0.709107,0.599303,-0.166667,0.334144,0.440379,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.634594


In [18]:
smiles1 = X1_all.getDF()["SMILES"]

In [36]:
smiles1 = pd.concat([X1_all.getDF()["SMILES"], X2_all.getDF()["SMILES"], X3_all.getDF()["SMILES"]]).reset_index()

In [62]:
X1_all2 = load_datasets("CK1/data/ck1_train_2")

X2_all2 = load_datasets("CK1/data/ck1_val_2")

X3_all2 = load_datasets("CK1/data/ck1_test_2")
smiles2 = pd.concat([X1_all2.getDF()["Drug"], X2_all2.getDF()["Drug"], X3_all2.getDF()["Drug"]]).reset_index()

In [53]:
smiles1[:X1_all.getDF().shape[0]]
smiles1[X1_all.getDF().shape[0]:-X3_all.getDF().shape[0]]
smiles1[-X3_all.getDF().shape[0]:]

,index,SMILES
488,0,COc1cc(C)c(NC(=O)CN(C)C)cc1Nc1nc(Nc2cccc(F)c2C...
489,1,CC(C)(C)c1cc(NC(=O)C(=O)c2cccc3ccccc23)n(-c2cc...
490,2,Nc1nnc(-c2cc3c(Oc4ccc(Cl)cc4)cncc3s2)o1
491,3,OCc1ccc(-c2nc(-c3ccccn3)c(-c3ccc4c(c3)OCO4)[nH...
492,4,CS(=O)(=O)CCNCc1ccoc1-c1ccc2ncnc(Nc3ccc(OCc4cc...
...,...,...
638,150,COc1cc2nc3nc(-c4cccs4)c(-c4cccs4)nc3nc2cc1OC
639,151,NC1(C(=O)NCc2ccc(Cl)cc2)CCN(c2ncnc3[nH]ccc23)CC1
640,152,CCc1cccc(NC(=O)Nc2ccc(Oc3ccc4nc(NC(=O)OC)[nH]c...
641,153,CC(C)CCN1c2nc(Nc3cc(F)c(O)c(F)c3)ncc2N(C)C(=O)C1C


In [125]:
smiles1["SMILES"]
X_all = pd.concat([X1_all.X, X2_all.X, X3_all.X])
y_all =  pd.concat([X1_all.y, X2_all.y, X3_all.y])
for i in range(2, 11):
    X1_all2 = load_datasets(f"CK1/data/ck1_train_{i}")
    X2_all2 = load_datasets(f"CK1/data/ck1_val_{i}")
    X3_all2 = load_datasets(f"CK1/data/ck1_test_{i}")
    smiles2 = pd.concat([X1_all2.getDF()["Drug"], X2_all2.getDF()["Drug"], X3_all2.getDF()["Drug"]]).reset_index()
    indices_map = [list(smiles1["SMILES"]).index(element) for element in smiles2["Drug"]]
    #display(y_all.iloc[indices_map].Y)
    #display(pd.concat([X1_all2.y, X2_all2.y, X3_all2.y]).Y)
    if np.array_equal(y_all.iloc[indices_map].Y, pd.concat([X1_all2.y, X2_all2.y, X3_all2.y]).Y):
        print("good")
    else: 
        print("Critical error")
        break
    X_res = X_all.iloc[indices_map]
    X1_res = X_res.iloc[:X1_all2.getDF().shape[0]]
    X2_res = X_res.iloc[X1_all2.getDF().shape[0]:-X3_all2.getDF().shape[0]]
    X3_res = X_res.iloc[-X3_all2.getDF().shape[0]:]
    X1_res.to_csv(f"CK1/mod_data/X1.{i}")
    X2_res.to_csv(f"CK1/mod_data/X2.{i}")
    X3_res.to_csv(f"CK1/mod_data/X3.{i}")
    X1_all2.y.to_csv(f"CK1/mod_data/y1.{i}")
    X2_all2.y.to_csv(f"CK1/mod_data/y2.{i}")
    X3_all2.y.to_csv(f"CK1/mod_data/y3.{i}")
    
    


good
good
good
good
good
good
good
good
good


In [ ]:
smiles2 = pd.concat([X1_all2.getDF()["Drug"], X2_all2.getDF()["Drug"], X3_all2.getDF()["Drug"]]).reset_index()
indices_map = [list(smiles1["SMILES"]).index(element) for element in smiles2["Drug"]]
print(indices_map) 

In [87]:
smiles2.iloc[3]

QSPRID                                      A2ARDataset_003
Drug      CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5ccc...
Name: 3, dtype: object

In [88]:
smiles1.iloc[643]

index                                                     0
SMILES    CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5ccc...
Name: 643, dtype: object

In [98]:

X_all = pd.concat([X1_all.X, X2_all.X, X3_all.X])
type(X_all.iloc[indices_map]

,300,301,302,303,304,305,306,307,308,309,...,RDkit_fr_piperdine,RDkit_fr_piperzine,RDkit_fr_pyridine,RDkit_fr_sulfide,RDkit_fr_sulfonamd,RDkit_fr_sulfone,RDkit_fr_thiazole,RDkit_fr_thiophene,RDkit_fr_unbrch_alkane,RDkit_qed
QSPRID,,,,,,,,,,,,,,,,,,,,,
A2ARDataset_000,-0.442472,0.207853,0.391181,0.222411,-0.084914,-0.652523,0.186761,-0.141471,-0.138574,0.327026,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.690353
A2ARDataset_001,-0.547015,0.326527,-0.055402,0.498394,0.059913,-0.783032,0.457208,-0.362948,0.182183,0.154815,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.488490
A2ARDataset_002,-0.183866,0.578967,0.161637,0.422820,-0.359473,-1.000239,0.778588,0.269597,0.671282,0.095040,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.389416
A2ARDataset_000,-0.441387,0.505895,0.160751,0.364176,-0.213932,-0.924353,0.151645,0.289251,0.624456,0.070934,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.169669
A2ARDataset_000,-0.388080,0.826816,0.427821,0.598129,-0.328920,-1.083285,0.688639,0.246900,0.361368,0.369830,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.231465
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
A2ARDataset_159,-0.577175,0.608860,0.091210,0.595915,0.356120,-0.719083,0.237224,-0.085256,-0.044304,0.147913,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.743948
A2ARDataset_468,-0.773678,0.317389,-0.069955,0.312257,-0.031650,-0.482003,0.612659,0.294355,0.184764,0.260350,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.434504
A2ARDataset_160,-0.444222,0.136241,0.316228,0.284731,-0.208043,-0.829347,0.444270,-0.389058,0.234052,0.321042,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.625114


In [116]:
y_all =  pd.concat([X1_all.y, X2_all.y, X3_all.y])
type(y_all.Y)

pandas.core.series.Series